# Architecture Sweep — Test 평가

이 노트북은 4개 구조 실험과 EXP03 fine-tuning 모델의 `checkpoint_best.pth`로 Test 추론과 평가를 순차 실행합니다.

- Test 입력: 1,010개 2.5D sample(각 sample은 3개 채널)
- 추론 기준: fold 0, `checkpoint_best.pth`, TTA 유지, sliding-window step 0.5
- 안전 설정: 전처리/저장 process 각각 1개, 모델 순차 실행
- 중단 복구: 이미 생성된 prediction은 `--continue_prediction`으로 건너뜀
- 주요 지표: 모든 Test pixel의 TP/FP/FN을 합산한 foreground Micro Dice

> Test 결과를 보고 모델, threshold 또는 후처리를 다시 선택하면 Test leakage가 됩니다. 모델 선택은 Validation 결과로 끝내고 Test는 최종 일반화 성능 보고에 사용하세요.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/home/stu03/projects/medical-cdss")
DATA_ROOT = Path("/home/stu03/medical-data")
SWEEP_ROOT = DATA_ROOT / "architecture_sweep"
RAW_DATASET = DATA_ROOT / "nnUNet_raw/Dataset001_BHSD_25D"
PREPROCESSED_DATASET = DATA_ROOT / "nnUNet_preprocessed/Dataset001_BHSD_25D"
TEST_IMAGES = RAW_DATASET / "imagesTs"
TEST_GT = RAW_DATASET / "evaluation_ground_truth"
PREDICTION_ROOT = SWEEP_ROOT / "test_predictions"
TRAINER_DIR = PROJECT_ROOT / "packages/bhsd_nnunet/src/bhsd_nnunet/trainers"
NNUNET_PREDICT = PROJECT_ROOT / ".venv/bin/nnUNetv2_predict"
NNUNET_EVALUATE = PROJECT_ROOT / ".venv/bin/nnUNetv2_evaluate_folder"
COMMON_CONFIG = PROJECT_ROOT / "training/configs/architecture_sweep_common.json"
FINETUNE_CONFIG = PROJECT_ROOT / "training/configs/exp03_more_conv_finetune.json"
FINETUNE_SOURCE = SWEEP_ROOT / "EXP03_more_conv_blocks/Dataset001_BHSD_25D/nnUNetTrainerBHSDMoreConvSweep__nnUNetPlans__2d/fold_0/checkpoint_best.pth"

PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)
print("Prediction root:", PREDICTION_ROOT)


## 모델 정의

`RUN_MODELS`에서 일부 이름만 남기면 해당 모델만 실행할 수 있습니다. 기본값은 5개 전체입니다.


In [ ]:
MODELS = {
    "EXP01_residual_encoder": {
        "result_root": SWEEP_ROOT / "EXP01_residual_encoder",
        "trainer": "nnUNetTrainerBHSDResidualEncoderSweep",
        "config": COMMON_CONFIG,
    },
    "EXP02_stage_attention": {
        "result_root": SWEEP_ROOT / "EXP02_stage_attention",
        "trainer": "nnUNetTrainerBHSDStageAttentionSweep",
        "config": COMMON_CONFIG,
    },
    "EXP03_more_conv_blocks": {
        "result_root": SWEEP_ROOT / "EXP03_more_conv_blocks",
        "trainer": "nnUNetTrainerBHSDMoreConvSweep",
        "config": COMMON_CONFIG,
    },
    "EXP04_dropout": {
        "result_root": SWEEP_ROOT / "EXP04_dropout",
        "trainer": "nnUNetTrainerBHSDDropoutSweep",
        "config": COMMON_CONFIG,
    },
    "EXP03_more_conv_blocks_finetune": {
        "result_root": SWEEP_ROOT / "EXP03_more_conv_blocks_finetune",
        "trainer": "nnUNetTrainerBHSDMoreConvFineTune",
        "config": FINETUNE_CONFIG,
    },
}

RUN_MODELS = list(MODELS)
RUN_MODELS


## 실행 전 검증

입력·정답 개수, 설정 파일, 각 best checkpoint를 확인합니다. 하나라도 없으면 추론을 시작하지 않습니다.


In [ ]:
def model_dir(spec):
    return (
        spec["result_root"]
        / "Dataset001_BHSD_25D"
        / f"{spec['trainer']}__nnUNetPlans__2d"
        / "fold_0"
    )

expected_cases = len(list(TEST_GT.glob("*.nii.gz")))
input_channel_files = len(list(TEST_IMAGES.glob("*.nii.gz")))
checks = []
for name in RUN_MODELS:
    spec = MODELS[name]
    checkpoint = model_dir(spec) / "checkpoint_best.pth"
    metrics_file = model_dir(spec) / "validation_metrics.json"
    checks.append({
        "model": name,
        "checkpoint_exists": checkpoint.is_file(),
        "checkpoint_gib": round(checkpoint.stat().st_size / 1024**3, 3) if checkpoint.is_file() else None,
        "validation_metrics_exists": metrics_file.is_file(),
        "config_exists": spec["config"].is_file(),
    })

check_df = pd.DataFrame(checks)
print(f"Test cases={expected_cases:,}, input channel files={input_channel_files:,}")
assert expected_cases > 0
assert input_channel_files == expected_cases * 3, "2.5D 입력은 case마다 3개 채널이어야 합니다."
assert check_df[["checkpoint_exists", "validation_metrics_exists", "config_exists"]].all().all(), check_df
display(check_df)


## Test 추론

다음 셀은 5개 모델을 **순차적으로** 실행합니다. 현재 서버 제한 때문에 병렬 실행하지 마세요.

- 출력: `/home/stu03/medical-data/architecture_sweep/test_predictions/<모델명>`
- 로그: 각 출력 폴더의 `predict.log`
- 중단 후 같은 셀을 다시 실행하면 완료된 prediction을 건너뜁니다.
- 셀 실행 중 `Kernel Interrupt`를 하면 현재 subprocess도 중단될 수 있습니다. 재실행하면 이어서 처리합니다.


In [ ]:
def prediction_count(output_dir: Path) -> int:
    return len(list(output_dir.glob("*.nii.gz")))

def run_prediction(name: str):
    spec = MODELS[name]
    output_dir = PREDICTION_ROOT / name
    output_dir.mkdir(parents=True, exist_ok=True)
    completed_before = prediction_count(output_dir)
    if completed_before >= expected_cases:
        print(f"[{name}] 이미 완료: {completed_before}/{expected_cases}")
        return

    env = os.environ.copy()
    env.update({
        "nnUNet_raw": str(DATA_ROOT / "nnUNet_raw"),
        "nnUNet_preprocessed": str(DATA_ROOT / "nnUNet_preprocessed"),
        "nnUNet_results": str(spec["result_root"]),
        "nnUNet_extTrainer": str(TRAINER_DIR),
        "BHSD_CONFIG_PATH": str(spec["config"]),
        "nnUNet_compile": "false",
        "nnUNet_n_proc_DA": "0",
        "PYTHONUNBUFFERED": "1",
    })
    if name == "EXP03_more_conv_blocks_finetune":
        env["BHSD_FINETUNE_CHECKPOINT"] = str(FINETUNE_SOURCE)

    command = [
        str(NNUNET_PREDICT),
        "-i", str(TEST_IMAGES),
        "-o", str(output_dir),
        "-d", "1",
        "-c", "2d",
        "-f", "0",
        "-tr", spec["trainer"],
        "-p", "nnUNetPlans",
        "-chk", "checkpoint_best.pth",
        "-step_size", "0.5",
        "-device", "cuda",
        "-npp", "1",
        "-nps", "1",
        "--continue_prediction",
    ]
    print(f"\n[{name}] 시작 ({completed_before}/{expected_cases} 완료 상태)")
    log_path = output_dir / "predict.log"
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\nCOMMAND: " + " ".join(command) + "\n")
        log.flush()
        process = subprocess.Popen(
            command, cwd=PROJECT_ROOT, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"{name} 추론 실패(return code={return_code}). 로그: {log_path}")
    completed_after = prediction_count(output_dir)
    if completed_after != expected_cases:
        raise RuntimeError(f"{name}: prediction {completed_after}/{expected_cases}")
    print(f"[{name}] 완료: {completed_after}/{expected_cases}")

for model_name in RUN_MODELS:
    run_prediction(model_name)

print("모든 선택 모델의 Test 추론이 완료되었습니다.")


## nnU-Net 기본 평가

각 prediction 폴더에 `summary.json`을 생성합니다. 이 요약의 Dice와 아래의 전체 Micro Dice는 집계 방식이 다를 수 있습니다.


In [ ]:
def run_official_evaluation(name: str):
    pred_dir = PREDICTION_ROOT / name
    if prediction_count(pred_dir) != expected_cases:
        raise RuntimeError(f"{name} prediction이 완료되지 않았습니다.")
    summary_path = pred_dir / "summary.json"
    command = [
        str(NNUNET_EVALUATE),
        str(TEST_GT),
        str(pred_dir),
        "-djfile", str(RAW_DATASET / "dataset.json"),
        "-pfile", str(PREPROCESSED_DATASET / "nnUNetPlans.json"),
        "-o", str(summary_path),
        "-np", "1",
    ]
    print(f"[{name}] official evaluation")
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    return summary_path

official_summaries = {name: run_official_evaluation(name) for name in RUN_MODELS}
official_summaries


## 전체 Test Micro Dice 계산

모든 Test sample의 foreground TP, FP, FN을 먼저 합산하고 다음 식으로 계산합니다.

`Micro Dice = 2TP / (2TP + FP + FN)`

추가로 precision, recall과 case별 Dice 평균도 함께 저장합니다.


In [ ]:
def safe_ratio(numerator: int, denominator: int) -> float:
    return float(numerator / denominator) if denominator else 1.0

def calculate_test_metrics(name: str):
    pred_dir = PREDICTION_ROOT / name
    gt_files = sorted(TEST_GT.glob("*.nii.gz"))
    tp = fp = fn = tn = 0
    case_rows = []

    for gt_path in gt_files:
        pred_path = pred_dir / gt_path.name
        if not pred_path.is_file():
            raise FileNotFoundError(f"Prediction 없음: {pred_path}")
        gt = np.asanyarray(nib.load(gt_path).dataobj) > 0
        pred = np.asanyarray(nib.load(pred_path).dataobj) > 0
        if gt.shape != pred.shape:
            raise ValueError(f"shape 불일치: {gt_path.name}, GT={gt.shape}, pred={pred.shape}")

        case_tp = int(np.logical_and(pred, gt).sum())
        case_fp = int(np.logical_and(pred, ~gt).sum())
        case_fn = int(np.logical_and(~pred, gt).sum())
        case_tn = int(np.logical_and(~pred, ~gt).sum())
        denominator = 2 * case_tp + case_fp + case_fn
        case_dice = safe_ratio(2 * case_tp, denominator)
        case_rows.append({
            "case": gt_path.name, "dice": case_dice,
            "tp": case_tp, "fp": case_fp, "fn": case_fn, "tn": case_tn,
        })
        tp += case_tp
        fp += case_fp
        fn += case_fn
        tn += case_tn

    micro_dice = safe_ratio(2 * tp, 2 * tp + fp + fn)
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    case_df = pd.DataFrame(case_rows)
    case_df.to_csv(pred_dir / "test_case_metrics.csv", index=False, encoding="utf-8-sig")
    return {
        "model": name,
        "test_cases": len(case_df),
        "test_micro_dice": micro_dice,
        "test_precision": precision,
        "test_recall": recall,
        "mean_case_dice": float(case_df["dice"].mean()),
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

results = [calculate_test_metrics(name) for name in RUN_MODELS]
result_df = pd.DataFrame(results).sort_values("test_micro_dice", ascending=False).reset_index(drop=True)
comparison_path = PREDICTION_ROOT / "test_metrics_comparison.csv"
result_df.to_csv(comparison_path, index=False, encoding="utf-8-sig")
display(result_df)
print("저장:", comparison_path)


## Validation과 Test 결과 확인

이 표와 그래프는 성능 보고용입니다. Test 열을 기준으로 모델이나 threshold를 다시 선택하지 마세요.


In [ ]:
validation_rows = []
for name in RUN_MODELS:
    metrics_path = model_dir(MODELS[name]) / "validation_metrics.json"
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    validation_rows.append({
        "model": name,
        "validation_micro_dice": metrics.get("best_validation_micro_dice"),
        "validation_best_epoch": metrics.get("best_epoch"),
    })

report_df = pd.DataFrame(validation_rows).merge(result_df, on="model", how="left")
report_path = PREDICTION_ROOT / "validation_test_report.csv"
report_df.to_csv(report_path, index=False, encoding="utf-8-sig")
display(report_df)

plot_df = report_df.set_index("model")[["validation_micro_dice", "test_micro_dice"]]
ax = plot_df.plot(kind="bar", figsize=(13, 5), ylim=(0, 1), rot=25)
ax.set_ylabel("Micro Dice")
ax.set_title("Validation vs Test Micro Dice")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
figure_path = PREDICTION_ROOT / "validation_test_micro_dice.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("표 저장:", report_path)
print("그래프 저장:", figure_path)


## 생성되는 결과

각 모델 폴더:

- `*.nii.gz`: 예측 segmentation
- `predict.log`: 추론 로그
- `summary.json`: nnU-Net 기본 평가 결과
- `test_case_metrics.csv`: case별 Dice/TP/FP/FN/TN

공통 폴더:

- `test_metrics_comparison.csv`: 5개 모델의 전체 Test Micro Dice 비교
- `validation_test_report.csv`: Validation/Test 통합 표
- `validation_test_micro_dice.png`: 비교 그래프

모든 결과는 `/home/stu03/medical-data/architecture_sweep/test_predictions`에 저장됩니다.
